# 03 – Baseline dan pembanding S1 sampai S4 (CP3)

- **S1:** Swin-T pralatih, weighted sampling, AdamW 1e-4, wd 0,05, batch 32, maks 30 epoch, henti dini sabar 5.
- **S2:** embedding Swin beku (sekali), SMOTE di dalam lipatan latih, kepala linear.
- **S3:** fitur PyRadiomics → MinMax → LASSO → SMOTE → SVM linear / LDA.
- **S4:** S3 + CTGAN pada ruang fitur (50% data latih). Analisis sensitivitas LASSO 0,01 berlabel eksploratif (D3).

Tiap job menyimpan `last.pt` (atomik) dan `DONE`; job selesai dilewati, job terputus dilanjutkan. Tiap job dijalankan sebagai pemanggilan fungsi dalam satu loop; memori GPU dibersihkan di akhir tiap job. **Jangan ganti seed atau buang lipatan karena hasil jelek.**

**Notebook mandiri (Versi 2):** tanpa folder `src`, tanpa Docker. Semua kode ada di sel-sel notebook ini. Kode ditulis dari Versi 1 dan **belum dijalankan** di mesin penulis (tanpa torch/GPU); jalankan dari atas ke bawah di mesin RTX 3070 Ti. Hasil yang tidak dapat dihitung dilaporkan apa adanya, bukan diisi.

## 1. Pustaka tambahan (opsional)

PyRadiomics di Windows kadang memerlukan kompiler C++ (MSVC Build Tools) atau roda siap pakai; bila gagal terpasang, S3 dan S4 tidak dapat dijalankan dan CP3 tetap `GAGAL` (tidak dipalsukan).

In [ ]:
INSTALL = False      # True hanya sekali, di venv yang sama dengan notebook 00 (torch CUDA sudah terpasang di sana)

if INSTALL:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-learn", "imbalanced-learn", "scipy", "pandas", "pyradiomics", "SimpleITK", "ctgan", "tabulate"])

## 2. Impor, konfigurasi, utilitas, perangkat, kebocoran, dan pemeriksaan checkpoint

In [ ]:
from __future__ import annotations

import contextlib, copy, datetime as dt, gc, hashlib, importlib, io, json, math, os, platform, random, re, shutil, subprocess, sys, tempfile, time
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import yaml
from IPython.display import display, Image

# expandable_segments mengurangi fragmentasi memori, tetapi belum didukung di Windows.
if platform.system() != "Windows":
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch
import torch.nn as nn
import torch.nn.functional as F


def find_root() -> Path:
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "configs" / "base.yaml").exists():
            return q
    raise FileNotFoundError("configs/base.yaml tidak ditemukan dari direktori kerja ke atas")


ROOT = find_root()
os.chdir(ROOT)

_ENV = {"LARYNX_DATASET": "dataset", "LARYNX_OUTPUTS": "outputs", "LARYNX_HF_CACHE": "hf_cache"}


def deep_merge(a: dict, b: dict) -> dict:
    out = copy.deepcopy(a)
    for k, v in b.items():
        out[k] = deep_merge(out[k], v) if isinstance(v, dict) and isinstance(out.get(k), dict) else copy.deepcopy(v)
    return out


def _load(path: Path) -> dict:
    with open(path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    base = cfg.pop("_base", None)
    return deep_merge(_load(path.parent / base), cfg) if base else cfg


def load_merged(path) -> dict:
    cfg = _load(Path(path))
    for env, key in _ENV.items():
        if os.environ.get(env):
            cfg["paths"][key] = os.environ[env]
    for key, val in cfg["paths"].items():           # jalur relatif -> mutlak terhadap folder Versi 2
        if not Path(val).is_absolute():
            cfg["paths"][key] = str((ROOT / val).resolve())
    return cfg


def P(cfg: dict, *parts: str) -> Path:
    return Path(cfg["paths"]["outputs"]).joinpath(*parts)


def config_hash(cfg: dict) -> str:
    return hashlib.sha256(json.dumps(cfg, sort_keys=True, ensure_ascii=False, default=str).encode("utf-8")).hexdigest()


def job_seed(base_seed: int, *parts) -> int:
    key = "|".join([str(base_seed), *map(str, parts)]).encode("utf-8")
    return int.from_bytes(hashlib.sha256(key).digest()[:4], "big")


def set_seed(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)


def atomic_write_bytes(path, data: bytes) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(dir=path.parent, prefix=path.name + ".", suffix=".tmp")
    try:
        with os.fdopen(fd, "wb") as f:
            f.write(data); f.flush(); os.fsync(f.fileno())
        os.replace(tmp, path)
    except BaseException:
        if os.path.exists(tmp):
            os.remove(tmp)
        raise


def atomic_write_json(path, obj) -> None:
    atomic_write_bytes(path, json.dumps(obj, indent=2, ensure_ascii=False, sort_keys=True, default=str).encode("utf-8"))


def atomic_torch_save(obj, path) -> None:
    buf = io.BytesIO(); torch.save(obj, buf); atomic_write_bytes(path, buf.getvalue())


def is_done(job_dir, cfg_hash: str) -> bool:
    done = Path(job_dir) / "DONE"
    return done.exists() and done.read_text(encoding="utf-8").strip() == cfg_hash


def mark_done(job_dir, cfg_hash: str) -> None:
    atomic_write_bytes(Path(job_dir) / "DONE", cfg_hash.encode("utf-8"))


def git_commit() -> str:
    # Versi 2 tanpa Docker: hash commit dari git (bila ada), ditambah -dirty bila ada perubahan belum di-commit.
    try:
        c = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "--short", "HEAD"], capture_output=True, text=True, timeout=15).stdout.strip()
        d = subprocess.run(["git", "-C", str(ROOT), "status", "--porcelain"], capture_output=True, text=True, timeout=15).stdout.strip()
        return (c or "tanpa-commit") + ("-dirty" if d and c else "")
    except Exception:  # noqa: BLE001
        return "tanpa-commit"


GIT_COMMIT = git_commit()
CFG_PATH = "configs/base.yaml"
cfg = load_merged(CFG_PATH)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR = P(cfg, "cp0")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("ROOT   :", ROOT)
print("dataset:", cfg["paths"]["dataset"])
print("outputs:", cfg["paths"]["outputs"])
print("device :", DEV, "| commit:", GIT_COMMIT, "| config_hash:", config_hash(cfg)[:12])

def free_gpu() -> None:
    # Pengganti 'proses baru per job' di Versi 1: dipanggil di akhir tiap job agar memori GPU kembali bersih.
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


WORK_DIR = Path(cfg["paths"]["outputs"]) / "_workers"


def install_module(name: str, src: str):
    # Windows memakai 'spawn': fungsi/kelas yang dipakai proses anak harus berada di modul yang dapat diimpor. Modul dibuat
    # saat notebook berjalan (satu-satunya sumber kodenya ada di sel notebook ini) lalu diimpor, termasuk oleh proses anak.
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    p = WORK_DIR / f"{name}.py"
    changed = (not p.exists()) or p.read_text(encoding="utf-8") != src
    if changed:
        p.write_text(src, encoding="utf-8")
    if str(WORK_DIR) not in sys.path:
        sys.path.insert(0, str(WORK_DIR))
    os.environ["PYTHONPATH"] = str(WORK_DIR) + os.pathsep + os.environ.get("PYTHONPATH", "")
    importlib.invalidate_caches()
    mod = importlib.import_module(name)
    return importlib.reload(mod) if changed and name in sys.modules else mod

In [ ]:
def configure_threads(cfg: dict) -> int:
    n = (cfg.get("compute") or {}).get("torch_threads") or os.cpu_count() or 1
    torch.set_num_threads(int(n))
    return int(n)


def use_amp(cfg: dict) -> bool:
    return bool((cfg.get("compute") or {}).get("amp", False)) and DEV == "cuda"


def autocast(cfg: dict):
    return torch.autocast(device_type="cuda", dtype=torch.float16) if use_amp(cfg) else contextlib.nullcontext()


def build_classifier(cfg: dict, pretrained: bool | None = None):
    from transformers import AutoConfig, AutoModelForImageClassification

    m = cfg["model"]
    pretrained = m["pretrained"] if pretrained is None else pretrained
    if pretrained:
        return AutoModelForImageClassification.from_pretrained(
            m["hf_name"], cache_dir=cfg["paths"]["hf_cache"], num_labels=m["num_classes"], ignore_mismatched_sizes=True)
    conf = AutoConfig.from_pretrained(m["hf_name"], cache_dir=cfg["paths"]["hf_cache"], num_labels=m["num_classes"])
    return AutoModelForImageClassification.from_config(conf)


def build_encoder(cfg: dict):
    from transformers import SwinModel

    return SwinModel.from_pretrained(cfg["model"]["hf_name"], cache_dir=cfg["paths"]["hf_cache"])


def num_workers(cfg: dict) -> int:
    v = cfg["loader"]["num_workers"]
    if v == "auto":
        return 0 if DEV == "cpu" else max(0, min(6, (os.cpu_count() or 2) - 1))
    return int(v)


def make_loader(ds, cfg: dict, batch_size: int, workers=None, **kw):
    # workers=0 untuk pembacaan evaluasi/embedding (tanpa augmentasi, murah): menghindari spawn proses berulang di Windows.
    from torch.utils.data import DataLoader

    nw = num_workers(cfg) if workers is None else int(workers)
    return DataLoader(ds, batch_size=batch_size, num_workers=nw, pin_memory=DEV == "cuda", persistent_workers=nw > 0, **kw)


@torch.no_grad()
def predict_probs(model, ds, batch_size: int, device: str, cfg: dict) -> np.ndarray:
    # Probabilitas kelas ganas (indeks 1) untuk tiap citra pada ds, urutan sama dengan ds.
    model.eval()
    out = []
    for x, _, _ in make_loader(ds, cfg, batch_size, workers=0, shuffle=False):
        with autocast(cfg):
            logits = model(pixel_values=x.to(device, non_blocking=True)).logits
        out.append(torch.softmax(logits.float(), dim=1)[:, 1].cpu().numpy())
    return np.concatenate(out) if out else np.zeros(0)


@torch.no_grad()
def embed_iter(encoder, ds, batch_size: int, device: str, cfg: dict):
    encoder.eval()
    for x, _, j in make_loader(ds, cfg, batch_size, workers=0, shuffle=False):
        with autocast(cfg):
            e = encoder(pixel_values=x.to(device, non_blocking=True)).pooler_output
        yield j.numpy(), e.float().cpu().numpy()

In [ ]:
# ---------------------------------------------------------------- checkpoint, persetujuan manusia, LOG.md
# Pengganti checkpoint.py dan approve.py Versi 1. Persetujuan MANUSIA tidak pernah dipanggil otomatis oleh notebook.
LULUS, GAGAL, MENUNGGU, TIDAK_TERSEDIA = "LULUS", "GAGAL", "MENUNGGU", "TIDAK_TERSEDIA"
DEC_PATH, BASE_PATH, LOG_PATH = ROOT / "configs" / "decisions.yaml", ROOT / "configs" / "base.yaml", ROOT / "LOG.md"


def file_sha256(path, chunk: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def write_manifest(out_dir, files, extra=None) -> Path:
    out_dir = Path(out_dir)
    entries = {}
    for f in files:
        p = Path(f)
        if not p.exists():
            raise FileNotFoundError(f"Keluaran checkpoint tidak ditemukan: {p}")
        key = str(p.relative_to(out_dir)) if p.is_relative_to(out_dir) else str(p)
        entries[key] = {"sha256": file_sha256(p), "bytes": p.stat().st_size}
    mp = out_dir / "manifest.json"
    atomic_write_json(mp, {"files": entries, **(extra or {})})
    return mp


def _item(nama: str, status: str, detail: str) -> dict:
    return {"item": nama, "status": status, "detail": detail}


def _review_item(out_dir) -> dict:
    out_dir = Path(out_dir)
    ok = (out_dir / "REVIEWED").exists()
    return _item("Tinjauan orang kedua (berkas REVIEWED)", LULUS if ok else MENUNGGU,
                 "ada" if ok else f"setelah meninjau hasil: approve_review('{out_dir.name.upper()}', 'NAMA') (RULES §3)")


def _flag_item(out_dir, name: str, nama: str, how: str) -> dict:
    ok = (Path(out_dir) / name).exists()
    return _item(nama, LULUS if ok else MENUNGGU, "ada" if ok else how)


def _decision_item(d: str) -> dict:
    with open(DEC_PATH, "r", encoding="utf-8") as f:
        dec = yaml.safe_load(f)[d]
    st = dec["status"]
    status = {"diputuskan": LULUS, "ditunda": LULUS, "menunggu_konfirmasi": MENUNGGU}.get(st, GAGAL)
    return _item(f"{d} {st}", status, str(dec.get("isi", "")))


def _finish(cp: str, out_dir, items: list, files: list, extra=None) -> dict:
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    blocking = [i for i in items if i["status"] in (GAGAL, MENUNGGU)]
    rep = {"checkpoint": cp, "lulus": not blocking, "items": items, "penghambat": [i["item"] for i in blocking]}
    atomic_write_json(out_dir / f"{cp.lower()}_check.json", rep)
    for i in items:
        print(f"[{i['status']:>15}] {i['item']} -- {i['detail']}")
    if rep["lulus"]:
        write_manifest(out_dir, [out_dir / f"{cp.lower()}_check.json", *[f for f in files if Path(f).exists()]],
                       extra={"checkpoint": cp, "git_commit": GIT_COMMIT, **(extra or {})})
        print(f"\n{cp} LULUS. manifest.json ditulis. Tag git '{cp.lower()}' dapat dibuat (setelah tinjauan orang kedua).")
    else:
        print(f"\n{cp} BELUM LULUS. Penghambat: {rep['penghambat']}\nmanifest.json TIDAK ditulis.")
    return rep


def log_row(who: str, what: str, hasil: str = "disetujui", keputusan: str = "via notebook") -> None:
    line = f"| {dt.date.today().isoformat()} | {who} | {what} | {hasil} | {keputusan} |\n"
    if LOG_PATH.exists():
        with open(LOG_PATH, "a", encoding="utf-8") as f:
            f.write(line)


def approve_flag(cp: str, name: str, by: str, what: str) -> None:
    d = P(load_merged(BASE_PATH), cp.lower())
    d.mkdir(parents=True, exist_ok=True)
    (d / name).write_text(f"{by} {dt.datetime.now().isoformat(timespec='seconds')}\n", encoding="utf-8")
    log_row(by, f"{what} ({cp})")
    print("ditulis", d / name)


def approve_review(cp: str, by: str) -> None:
    approve_flag(cp, "REVIEWED", by, "Tinjauan orang kedua")


def approve_decisions(ids: list, by: str) -> None:
    text = DEC_PATH.read_text(encoding="utf-8")
    for d in ids:
        pat = re.compile(rf"(^{d}:\n  status: )menunggu_konfirmasi", re.M)
        if not pat.search(text):
            print(f"{d}: tidak berstatus menunggu_konfirmasi (dilewati)")
            continue
        text = pat.sub(rf"\1diputuskan\n  dikonfirmasi_oleh: {by}\n  tanggal_konfirmasi: {dt.date.today().isoformat()}", text)
        log_row(by, f"Konfirmasi keputusan {d}")
        print(f"{d}: diputuskan oleh {by}")
    DEC_PATH.write_text(text, encoding="utf-8")


def approve_roi(by: str) -> None:
    t = BASE_PATH.read_text(encoding="utf-8")
    t2 = re.sub(r"(  roi:.*?\n    locked: )false", r"\1true", t, count=1, flags=re.S)
    if t2 == t and "locked: true" not in t:
        raise SystemExit("preprocess.roi.locked tidak ditemukan di base.yaml")
    BASE_PATH.write_text(t2, encoding="utf-8")
    d = P(load_merged(BASE_PATH), "cp2")
    d.mkdir(parents=True, exist_ok=True)
    (d / "ROI_APPROVED").write_text(f"{by} {dt.datetime.now().isoformat(timespec='seconds')}\n", encoding="utf-8")
    log_row(by, "Parameter Frangi dikunci setelah pemeriksaan visual 50 citra (FR-03)")
    print("preprocess.roi.locked=true; ditulis", d / "ROI_APPROVED")

In [ ]:
# ---------------------------------------------------------------- pengendalian kebocoran L1-L6 (dari leakage.py Versi 1)
class LeakageError(AssertionError):
    pass


def check_l1(splits: dict) -> bool:
    # L1: tidak ada pasien di latih/validasi dan uji sekaligus; tiap pasien diuji tepat sekali per ulangan.
    allp = set(splits["all_patients"])
    for r in splits["repeats"]:
        tested = []
        for f in r["folds"]:
            tr, va, te = set(f["train"]), set(f["val"]), set(f["test"])
            if tr & te:
                raise LeakageError(f"L1: pasien latih juga di uji (rep {r['rep']} lipatan {f['fold']}): {sorted(tr & te)[:5]}")
            if va & te:
                raise LeakageError(f"L1: pasien validasi juga di uji (rep {r['rep']} lipatan {f['fold']}): {sorted(va & te)[:5]}")
            if tr & va:
                raise LeakageError(f"L1: pasien latih juga di validasi (rep {r['rep']} lipatan {f['fold']})")
            if tr | va | te != allp:
                raise LeakageError(f"L1: gabungan partisi != seluruh pasien (rep {r['rep']} lipatan {f['fold']})")
            tested += list(te)
        if sorted(tested) != sorted(allp):
            raise LeakageError(f"L1: pada rep {r['rep']} tiap pasien harus diuji tepat sekali")
    return True


def check_l5(splits: dict, resolution_of: dict) -> bool:
    # L5 (S7): semua pasien latih/validasi berresolusi mayoritas, semua pasien uji bukan.
    major = splits["major_resolution"]
    for r in splits["repeats"]:
        for f in r["folds"]:
            tr, va, te = set(f["train"]), set(f["val"]), set(f["test"])
            if (tr | va) & te or tr & va:
                raise LeakageError(f"L5: partisi S7 saling tumpang tindih (rep {r['rep']})")
            bad = [p for p in tr | va if resolution_of[p] != major]
            if bad:
                raise LeakageError(f"L5: pasien non-mayoritas di latih/validasi: {sorted(bad)[:5]}")
            bad = [p for p in te if resolution_of[p] == major]
            if bad:
                raise LeakageError(f"L5: pasien resolusi mayoritas di uji S7: {sorted(bad)[:5]}")
    return True


def check_splits(splits: dict, resolution_of=None) -> bool:
    if splits.get("kind") == "s7":
        if resolution_of is None:
            raise LeakageError("L5: peta resolusi pasien diperlukan untuk memeriksa split S7")
        return check_l5(splits, resolution_of)
    return check_l1(splits)


def check_l6(cfg: dict) -> bool:
    # L6: metrik struktur pembuluh memakai filter berbeda dari pembuat ROI.
    if cfg["gan"]["quality"]["vessel_filter"].lower() == cfg["preprocess"]["roi"]["filter"].lower():
        raise LeakageError("L6: filter metrik pembuluh sama dengan filter pembuat ROI")
    return True


def check_l3_job(job_patients: dict, split_fold: dict, synthetic_source_patients=None) -> bool:
    # L3: pasien yang dipakai job sama dengan partisi split; citra sintetis hanya bersumber dari pasien latih.
    test = set(split_fold["test"])
    used = set(job_patients["train"]) | set(job_patients["val"])
    if used & test:
        raise LeakageError(f"L3: job memakai pasien uji: {sorted(used & test)[:5]}")
    if set(job_patients["train"]) != set(split_fold["train"]) or set(job_patients["val"]) != set(split_fold["val"]):
        raise LeakageError("L3: daftar pasien job berbeda dari berkas split")
    if set(job_patients["test"]) != test:
        raise LeakageError("L3: daftar pasien uji job berbeda dari berkas split")
    if synthetic_source_patients is not None and not set(synthetic_source_patients) <= set(split_fold["train"]):
        raise LeakageError("L3: citra sintetis bersumber dari pasien non-latih")
    return True


def check_l3_dir(jobs_root, scenario: str, splits: dict) -> int:
    n = 0
    for r in splits["repeats"]:
        for f in r["folds"]:
            pj = Path(jobs_root) / scenario / f"rep{r['rep']}_fold{f['fold']}" / "patients.json"
            if pj.exists():
                check_l3_job(json.loads(pj.read_text(encoding="utf-8")), f)
                n += 1
    return n


class GuardedArray:
    # L2: pembungkus larik yang menolak akses ke baris terlarang (indeks uji) dan penyerahan seluruh larik.
    def __init__(self, arr, forbidden):
        self._arr = arr
        self._forbidden = np.asarray(sorted(set(map(int, forbidden))), dtype=np.int64)
        self.accessed = set()

    def __len__(self):
        return len(self._arr)

    @property
    def shape(self):
        return self._arr.shape

    def __array__(self, *a, **k):
        raise LeakageError("L2: seluruh larik diminta pada fase fit; gunakan indeks latih/validasi saja")

    def __getitem__(self, idx):
        if isinstance(idx, slice):
            pos = np.arange(len(self._arr))[idx]
        else:
            idx_a = np.asarray(idx)
            pos = np.flatnonzero(idx_a) if idx_a.dtype == bool else idx_a.ravel().astype(np.int64)
        if np.intersect1d(pos, self._forbidden).size:
            raise LeakageError("L2: fase fit mengakses baris data uji")
        self.accessed.update(int(p) for p in pos)
        return np.asarray(self._arr[idx])

In [ ]:
# ---------------------------------------------------------------- keluaran job, pembacaan split, penjalan job (dari jobio.py, splits.py, jobs.py)
def job_dir(cfg: dict, scenario: str, rep: int, fold: int) -> Path:
    return P(cfg, "jobs", scenario, f"rep{rep}_fold{fold}")


def resolution_map(cfg: dict) -> dict:
    a = pd.read_csv(P(cfg, "cp1", "audit.csv"))
    return dict(zip(a.patient_uid, a.resolution))


def write_run_info(d: Path, cfg: dict, cfg_hash: str, seed: int, extra: dict) -> None:
    atomic_write_json(d / "run_info.json", {
        "scenario": cfg.get("scenario"), "config_hash": cfg_hash, "seed": seed, "git_commit": GIT_COMMIT,
        "torch": torch.__version__, "platform": platform.platform(), **extra})


def write_preds(d: Path, name: str, meta: pd.DataFrame, idx: np.ndarray, probs: np.ndarray, thr: float) -> pd.DataFrame:
    # preds_<name>_images.csv (per citra) dan preds_<name>_patients.csv (rata-rata per pasien, FR-09)
    df = pd.DataFrame({"idx": idx, "patient_uid": meta.patient_uid.to_numpy()[idx],
                       "label": meta.label.to_numpy()[idx], "prob_malignant": probs})
    atomic_write_bytes(d / f"preds_{name}_images.csv", df.to_csv(index=False).encode("utf-8"))
    pat = df.groupby("patient_uid").agg(label=("label", "first"), prob_malignant=("prob_malignant", "mean"),
                                        n_images=("idx", "size")).reset_index()
    pat["pred"] = (pat.prob_malignant >= thr).astype(int)
    atomic_write_bytes(d / f"preds_{name}_patients.csv", pat.to_csv(index=False).encode("utf-8"))
    return pat


def balanced_logloss(y: np.ndarray, p1: np.ndarray, eps: float = 1e-7) -> float:
    p1 = np.clip(p1, eps, 1 - eps)
    losses = [(-np.log(1 - p1[y == 0])).mean() if (y == 0).any() else None,
              (-np.log(p1[y == 1])).mean() if (y == 1).any() else None]
    return float(np.mean([v for v in losses if v is not None]))


# ---------------------------------------------------------------- split: baca
def _split_name(splits_name: str):
    return ("splits.json", "fold_summary.csv") if splits_name == "main" else (f"splits_{splits_name}.json", f"fold_summary_{splits_name}.csv")


def load_splits(cfg: dict, splits_name=None) -> dict:
    fj, _ = _split_name(splits_name or cfg.get("splits_name", "main"))
    return json.loads(P(cfg, "splits", fj).read_text(encoding="utf-8"))


def get_fold(splits: dict, rep: int, fold: int) -> dict:
    return splits["repeats"][rep]["folds"][fold]


# ---------------------------------------------------------------- penjalan job (pengganti jobs.py: pemanggilan fungsi, bukan subproses)
def job_status(cfg: dict, scenario: str) -> list:
    rows = []
    for r in load_splits(cfg)["repeats"]:
        for f in r["folds"]:
            d = job_dir(cfg, scenario, r["rep"], f["fold"])
            rows.append({"rep": r["rep"], "fold": f["fold"], "DONE": (d / "DONE").exists(), "ada_last_pt": (d / "last.pt").exists()})
    return rows


def _selected(splits: dict, reps, folds):
    for r in splits["repeats"]:
        if reps is not None and r["rep"] not in reps:
            continue
        for f in r["folds"]:
            if folds is None or f["fold"] in folds:
                yield r["rep"], f["fold"]


def run_job_safe(label: str, fn, cfg: dict, attempts: int = 2):
    """Jalankan satu job. Hanya RuntimeError (termasuk CUDA OOM) diulang sekali setelah memori GPU dibersihkan; job yang gagal
    dua kali menghentikan antrean dan HARUS dicatat di LOG.md (RULES §4). Tidak ada seed diganti atau lipatan dibuang."""
    log = P(cfg, "jobs", "launch.log")
    log.parent.mkdir(parents=True, exist_ok=True)
    err = None
    for a in range(1, attempts + 1):
        t0 = time.time()
        try:
            res = fn()
            ok = True
        except RuntimeError as e:
            res, ok, err = None, False, repr(e)[:300]
        free_gpu()
        with open(log, "a", encoding="utf-8") as fh:
            fh.write(f"{dt.datetime.now().isoformat(timespec='seconds')}\t{label}\tpercobaan={a}\t{'OK' if ok else 'GAGAL ' + str(err)}\t{time.time() - t0:.0f}s\n")
        if ok:
            return res
    raise RuntimeError(f"Job gagal {attempts}x berturut-turut: {label}. Antrean dihentikan. Catat penyebabnya di LOG.md sebelum dilanjutkan (RULES §4). Terakhir: {err}")


def launch_scenario(config: str, fn, reps=None, folds=None) -> None:
    # satu job per (rep, lipatan): fn(cfg, rep, fold)
    cfg = load_merged(config)
    for r, k in _selected(load_splits(cfg), reps, folds):
        run_job_safe(f"{cfg['scenario']}/rep{r}_fold{k}", lambda r=r, k=k: fn(load_merged(config), r, k), cfg)


def launch_gan_chain(config: str, gan_fn, clf_fn, ratios=None, reps=None, folds=None) -> None:
    # S5/S6/S7: GAN lipatan k, lalu klasifikator tiap rasio (urutan wajib, PRD §8). ratios=[None] untuk S7.
    cfg = load_merged(config)
    sc = cfg["scenario"]
    ratios = ratios if ratios is not None else cfg["gan"]["ratios"]
    for r, k in _selected(load_splits(cfg), reps, folds):
        run_job_safe(f"{sc}_gan/rep{r}_fold{k}", lambda r=r, k=k: gan_fn(load_merged(config), r, k), cfg)
        for ra in ratios:
            def _clf(r=r, k=k, ra=ra):
                c = load_merged(config)
                if ra is not None:
                    c["synthetic"]["ratio"] = ra
                return clf_fn(c, r, k)
            tag = f"_r{int(round(ra * 100))}" if ra is not None else ""
            run_job_safe(f"{sc}{tag}/rep{r}_fold{k}", _clf, cfg)

## 3. Pembacaan cache dan Dataset (dari `data.py`; modul `larynx_data` dibuat saat sel dijalankan agar DataLoader multi-proses berfungsi di Windows)

In [ ]:
# ---------------------------------------------------------------- pembacaan cache memmap + Dataset (dari data.py Versi 1)
DATA_SRC = r'''
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from torchvision.transforms import v2


class Cache:
    # images/roi dibuka MALAS dengan memmap dan TIDAK ikut ter-pickle: pickle np.memmap menyalin seluruh isinya
    # (2,2 GB x jumlah pekerja DataLoader di Windows/spawn). Tiap proses membuka ulang memmap sendiri.
    def __init__(self, cache_dir):
        self.dir = Path(cache_dir)
        self.hash = (self.dir / "DONE").read_text(encoding="utf-8").strip()
        self.meta = pd.read_csv(self.dir / "meta.csv")
        self._images = None
        self._roi = None

    @property
    def images(self):
        if self._images is None:
            self._images = np.load(self.dir / "images.npy", mmap_mode="r")
        return self._images

    @property
    def roi(self):
        if self._roi is None:
            self._roi = np.load(self.dir / "roi.npy", mmap_mode="r")
        return self._roi

    def __getstate__(self):
        st = self.__dict__.copy()
        st["_images"] = None
        st["_roi"] = None
        return st

    def indices_of(self, patients):
        return self.meta.index[self.meta.patient_uid.isin(set(patients))].to_numpy()

    @property
    def labels(self):
        return self.meta.label.to_numpy()


def build_transform(cfg, train):
    size, aug, pp = cfg["model"]["input_size"], cfg["augment"], cfg["preprocess"]
    t = []
    if train:
        t.append(v2.RandomResizedCrop(size, scale=tuple(aug["crop_scale"]), ratio=tuple(aug["crop_ratio"]), antialias=True))
        if aug["hflip"]:
            t.append(v2.RandomHorizontalFlip())
        if aug["vflip"]:
            t.append(v2.RandomVerticalFlip())
        if aug["rotation_deg"]:
            t.append(v2.RandomRotation(aug["rotation_deg"]))
    else:
        t.append(v2.Resize((size, size), antialias=True))
    t += [v2.ToDtype(torch.float32, scale=True), v2.Normalize(pp["imagenet_mean"], pp["imagenet_std"])]
    return v2.Compose(t)


class ImageDS(Dataset):
    # (tensor ternormalisasi ImageNet, label, indeks cache), dilabeli ganas dan indeks -1 untuk sintetis.
    # synthetic: None | array uint8 (N,H,W,3) | (jalur .npy, indeks terpilih): bentuk terakhir dibuka dengan memmap di tiap
    # proses (tidak ikut ter-pickle ke pekerja DataLoader).
    def __init__(self, cache, idx, cfg, train, synthetic=None):
        self.cache, self.idx = cache, np.asarray(idx)
        self.syn, self.syn_path, self.syn_idx, self._syn_mm = None, None, None, None
        if isinstance(synthetic, tuple):
            self.syn_path, self.syn_idx = str(synthetic[0]), np.asarray(synthetic[1])
            n_syn = len(self.syn_idx)
        elif synthetic is not None:
            self.syn = synthetic
            n_syn = len(synthetic)
        else:
            n_syn = 0
        self.n_real = len(self.idx)
        self.tf = build_transform(cfg, train)
        self.labels = np.concatenate([cache.labels[self.idx], np.ones(n_syn, dtype=cache.labels.dtype)])

    def __getstate__(self):
        st = self.__dict__.copy()
        st["_syn_mm"] = None
        return st

    def _synthetic(self, k):
        if self.syn_path is None:
            return self.syn[k]
        if self._syn_mm is None:
            self._syn_mm = np.load(self.syn_path, mmap_mode="r")
        return self._syn_mm[int(self.syn_idx[k])]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        if i < self.n_real:
            j = int(self.idx[i])
            arr = np.array(self.cache.images[j])
        else:
            j = -1
            arr = np.array(self._synthetic(i - self.n_real))
        x = torch.from_numpy(arr).permute(2, 0, 1).contiguous()
        return self.tf(x), int(self.labels[i]), j
'''

larynx_data = install_module("larynx_data", DATA_SRC)
Cache, ImageDS = larynx_data.Cache, larynx_data.ImageDS


def find_cache(cfg: dict) -> Path:
    cur = P(cfg, "cache", "current.json")
    if not cur.exists():
        raise FileNotFoundError("Cache belum dibuat (jalankan notebook 02)")
    return Path(json.loads(cur.read_text(encoding="utf-8"))["dir"])

## 4. Kode pelatihan: S1 (Swin-T), S2 (embedding), S3/S4 (radiomik), evaluasi

In [ ]:
# ---------------------------------------------------------------- klasifikator Swin-T (dari train.py Versi 1): S1, S5, S6, S7
from sklearn.metrics import roc_auc_score
from torch.utils.data import WeightedRandomSampler


def resolve_ratio(cfg: dict):
    syn = cfg.get("synthetic")
    if not syn:
        return None
    r = syn.get("ratio")
    if r == "best":
        raise ValueError("ratio 'best' dihapus di Versi 2: konfigurasi CV utama melibatkan pasien uji S7 (L5). Gunakan select_s7_ratio.")
    if r is None:
        raise ValueError("synthetic.ratio belum diisi (peluncur mengisinya per rasio)")
    return float(r)


def scenario_name(cfg: dict) -> str:
    syn = cfg.get("synthetic")
    if syn and syn.get("ratio") is not None:
        return f"{cfg['scenario']}_r{int(round(float(syn['ratio']) * 100))}"
    return cfg["scenario"]


def load_synthetic(cfg: dict, rep: int, fold: int, sf: dict):
    # Subset citra sintetis (bersarang: awalan dari urutan acak tetap) + pemeriksaan L3.
    ratio = resolve_ratio(cfg)
    if ratio is None:
        return None, {}
    gdir = job_dir(cfg, f"{cfg['synthetic']['gan_scenario']}_gan", rep, fold)
    if not (gdir / "DONE").exists():
        raise FileNotFoundError(f"GAN {gdir} belum DONE; jalankan job GAN lebih dulu (urutan wajib, PRD §8)")
    meta = json.loads((gdir / "synth_meta.json").read_text(encoding="utf-8"))
    check_l3_job({"train": sf["train"], "val": sf["val"], "test": sf["test"]}, sf, synthetic_source_patients=meta["source_patients"])
    spath = gdir / "synth.npy"
    n_gen = int(np.load(spath, mmap_mode="r").shape[0])
    perm = np.random.default_rng(job_seed(cfg["seed"], "synorder", rep, fold)).permutation(n_gen)
    k = int(round(ratio * n_gen))
    return (spath, perm[:k]), {"ratio": ratio, "n_synthetic": int(k), "n_generated": n_gen, "gan_dir": str(gdir)}


def _patient_arrays(cache, idx: np.ndarray, probs: np.ndarray):
    df = pd.DataFrame({"p": cache.meta.patient_uid.to_numpy()[idx], "y": cache.labels[idx], "s": probs})
    g = df.groupby("p").agg(y=("y", "first"), s=("s", "mean"))
    return g.y.to_numpy(), g.s.to_numpy()


def run_swin_job(cfg: dict, rep: int, fold: int, cache_dir=None) -> str:
    scenario = scenario_name(cfg)
    splits = load_splits(cfg)
    check_splits(splits, resolution_map(cfg) if splits.get("kind") == "s7" else None)     # L1/L5 sebelum job apa pun
    cache = Cache(cache_dir or find_cache(cfg))
    d = job_dir(cfg, scenario, rep, fold)
    h = config_hash({"cfg": cfg, "rep": rep, "fold": fold, "cache": cache.hash, "splits": splits["meta"]["hash"]})
    if is_done(d, h):
        print(f"[dilewati] {scenario} rep{rep} fold{fold} sudah DONE")
        return str(d)
    d.mkdir(parents=True, exist_ok=True)

    sf = get_fold(splits, rep, fold)
    atomic_write_json(d / "patients.json", {"train": sf["train"], "val": sf["val"], "test": sf["test"]})
    check_l3_job(json.loads((d / "patients.json").read_text(encoding="utf-8")), sf)         # L3
    syn_arr, syn_info = load_synthetic(cfg, rep, fold, sf)

    seed = job_seed(cfg["seed"], scenario, rep, fold)
    set_seed(seed)
    threads = configure_threads(cfg)
    device = DEV
    tr = cfg["train"]
    amp = use_amp(cfg)

    idx_tr, idx_va = cache.indices_of(sf["train"]), cache.indices_of(sf["val"])
    ds_tr = ImageDS(cache, idx_tr, cfg, True, synthetic=syn_arr)
    ds_va = ImageDS(cache, idx_va, cfg, False)
    y_tr = ds_tr.labels
    cls_count = np.bincount(y_tr, minlength=2).astype(float)
    weights = (1.0 / np.maximum(cls_count, 1))[y_tr] if tr["weighted_sampling"] else np.ones(len(y_tr))

    model = build_classifier(cfg).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=tr["lr"], weight_decay=tr["weight_decay"])
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    lossf = torch.nn.CrossEntropyLoss()

    st = {"epoch": 0, "best": float("inf"), "bad": 0, "history": []}
    last = d / "last.pt"
    if last.exists():
        ck = torch.load(last, map_location="cpu", weights_only=False)
        if ck.get("config_hash") == h:
            model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
            if ck.get("scaler"):
                scaler.load_state_dict(ck["scaler"])
            st = {k: ck[k] for k in ("epoch", "best", "bad", "history")}
            torch.set_rng_state(ck["rng_torch"]); np.random.set_state(ck["rng_np"]); random.setstate(ck["rng_py"])
            print(f"[lanjut] dari epoch {st['epoch']}")
        else:
            print("[peringatan] last.pt dari konfigurasi lain diabaikan")

    write_run_info(d, cfg, h, seed, {"rep": rep, "fold": fold, "cache_hash": cache.hash, "scenario_job": scenario,
                                     "splits_hash": splits["meta"]["hash"], "splits_kind": splits.get("kind"),
                                     "device": device, "amp": amp, "torch_threads": threads,
                                     "n_train_real_images": int(len(idx_tr)), "n_val_images": int(len(idx_va)), **syn_info})

    # Satu DataLoader untuk seluruh job (pekerja persisten); sampler di-seed ulang tiap epoch = seed + epoch (sama dengan Versi 1).
    gsamp = torch.Generator()
    sampler = WeightedRandomSampler(torch.as_tensor(weights, dtype=torch.double), num_samples=len(ds_tr), replacement=True, generator=gsamp)
    loader = make_loader(ds_tr, cfg, tr["batch_size"], sampler=sampler)

    while st["epoch"] < tr["max_epochs"] and st["bad"] < tr["early_stop_patience"]:
        ep, t0 = st["epoch"], time.time()
        gsamp.manual_seed(seed + ep)
        model.train()
        tot, n = 0.0, 0
        for x, y, _ in loader:
            opt.zero_grad(set_to_none=True)
            with autocast(cfg):
                logits = model(pixel_values=x.to(device, non_blocking=True)).logits
            loss = lossf(logits.float(), y.to(device, non_blocking=True))
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item() * len(y)
            n += len(y)
        pv = predict_probs(model, ds_va, tr["batch_size"], device, cfg)
        val_loss = balanced_logloss(ds_va.labels, pv)
        try:
            pat_auc = float(roc_auc_score(*_patient_arrays(cache, idx_va, pv)))
        except ValueError:
            pat_auc = float("nan")
        st["history"].append({"epoch": ep + 1, "train_loss": tot / max(n, 1), "val_loss_balanced": val_loss,
                              "val_auc_patient": pat_auc, "seconds": time.time() - t0})
        print(st["history"][-1], flush=True)
        if val_loss < st["best"] - tr["min_delta"]:
            st["best"], st["bad"] = val_loss, 0
            atomic_torch_save(model.state_dict(), d / "best.pt")
        else:
            st["bad"] += 1
        st["epoch"] = ep + 1
        if st["epoch"] % tr["checkpoint_every_epochs"] == 0:
            atomic_torch_save({"model": model.state_dict(), "opt": opt.state_dict(), "config_hash": h,
                               "scaler": scaler.state_dict() if amp else None, "rng_torch": torch.get_rng_state(),
                               "rng_np": np.random.get_state(), "rng_py": random.getstate(), **st}, last)

    # ---- pelatihan selesai; baru sekarang data uji disentuh
    model.load_state_dict(torch.load(d / "best.pt", map_location="cpu"))
    thr = cfg["eval"]["threshold"]
    write_preds(d, "val", cache.meta, idx_va, predict_probs(model, ds_va, tr["batch_size"], device, cfg), thr)
    idx_te = cache.indices_of(sf["test"])
    ds_te = ImageDS(cache, idx_te, cfg, False)
    write_preds(d, "test", cache.meta, idx_te, predict_probs(model, ds_te, tr["batch_size"], device, cfg), thr)
    atomic_write_bytes(d / "history.csv", pd.DataFrame(st["history"]).to_csv(index=False).encode("utf-8"))
    mark_done(d, h)
    del loader, model, opt
    return str(d)

In [ ]:
# ---------------------------------------------------------------- S2: embedding Swin beku + SMOTE + regresi logistik (embed.py, train_s2.py)
from imblearn.over_sampling import SMOTE
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import Lasso, LogisticRegression
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.svm import SVC


def _emb_key(cfg: dict, cache) -> dict:
    # semua yang mengubah nilai embedding: model, ukuran masukan, normalisasi (mean DAN std), presisi, cache
    return {"model": cfg["model"], "normalize": cfg["preprocess"]["normalize"], "mean": cfg["preprocess"]["imagenet_mean"],
            "std": cfg["preprocess"]["imagenet_std"], "amp": use_amp(cfg), "cache": cache.hash}


def embedding_path(cfg: dict, cache) -> Path:
    return P(cfg, "embeddings", f"emb_{config_hash(_emb_key(cfg, cache))[:12]}")


def run_embed(cfg: dict, cache_dir=None) -> Path:
    # Tidak ada fit pada data di sini (bobot pralatih, normalisasi ImageNet tetap) -> aman dipakai semua lipatan (L2). Dapat dilanjutkan.
    cache = Cache(cache_dir or find_cache(cfg))
    d = embedding_path(cfg, cache)
    h = config_hash(_emb_key(cfg, cache))
    if is_done(d, h):
        print(f"[dilewati] embedding {d.name} sudah DONE")
        return d
    d.mkdir(parents=True, exist_ok=True)
    set_seed(cfg["seed"])
    configure_threads(cfg)
    enc = build_encoder(cfg).to(DEV)
    N, dim = len(cache.meta), int(enc.num_features)
    f, prog = d / "emb.npy", d / "progress.json"
    done_n = json.loads(prog.read_text())["done"] if prog.exists() and f.exists() else 0
    mm = np.lib.format.open_memmap(f, mode="r+" if f.exists() and done_n else "w+", dtype=np.float32, shape=(N, dim))
    bs = cfg["s2"]["embed_batch_size"]
    chunk = bs * 8
    for s in range(done_n, N, chunk):
        idx = np.arange(s, min(s + chunk, N))
        for j, e in embed_iter(enc, ImageDS(cache, idx, cfg, train=False), bs, DEV, cfg):
            mm[j] = e
        mm.flush()
        atomic_write_json(prog, {"done": int(idx[-1] + 1), "total": N})
        print(f"  embedding {idx[-1] + 1}/{N}", flush=True)
    del mm, enc
    free_gpu()
    mark_done(d, h)
    return d


def s2_fit_fold(X, y, idx_train, idx_val, s2: dict, seed: int) -> dict:
    # Fase fit. X dan y WAJIB GuardedArray yang melarang indeks uji. C dipilih dari val loss seimbang pasien validasi (L4).
    Xtr, ytr = X[idx_train], y[idx_train]
    scaler = StandardScaler().fit(Xtr)
    Xs = scaler.transform(Xtr)
    k = max(1, min(s2["smote_k"], int(np.bincount(ytr).min()) - 1))
    Xres, yres = SMOTE(sampling_strategy=s2["smote_strategy"], k_neighbors=k, random_state=seed).fit_resample(Xs, ytr)
    Xva, yva = scaler.transform(X[idx_val]), y[idx_val]
    best = None
    for C in s2["c_grid"]:
        clf = LogisticRegression(C=C, max_iter=s2["max_iter"]).fit(Xres, yres)
        vl = balanced_logloss(yva, clf.predict_proba(Xva)[:, 1])
        if best is None or vl < best["val_loss"]:
            best = {"C": C, "val_loss": vl, "clf": clf}
    return {"scaler": scaler, "clf": best["clf"], "C": best["C"], "val_loss": best["val_loss"],
            "n_train": int(len(ytr)), "n_after_smote": int(len(yres))}


def s2_predict(fit: dict, X, idx) -> np.ndarray:
    return fit["clf"].predict_proba(fit["scaler"].transform(X[idx]))[:, 1]


def run_s2_job(cfg: dict, rep: int, fold: int, cache_dir=None) -> str:
    scenario = cfg["scenario"]
    splits = load_splits(cfg)
    check_l1(splits)
    cache = Cache(cache_dir or find_cache(cfg))
    ed = embedding_path(cfg, cache)
    d = job_dir(cfg, scenario, rep, fold)
    h = config_hash({"cfg": cfg, "rep": rep, "fold": fold, "cache": cache.hash, "splits": splits["meta"]["hash"]})
    if is_done(d, h):
        print(f"[dilewati] {scenario} rep{rep} fold{fold} sudah DONE")
        return str(d)
    if not (ed / "DONE").exists():
        raise FileNotFoundError("Embedding belum selesai (jalankan run_embed)")
    d.mkdir(parents=True, exist_ok=True)
    sf = get_fold(splits, rep, fold)
    atomic_write_json(d / "patients.json", {"train": sf["train"], "val": sf["val"], "test": sf["test"]})
    check_l3_job(json.loads((d / "patients.json").read_text(encoding="utf-8")), sf)
    seed = job_seed(cfg["seed"], scenario, rep, fold)
    set_seed(seed)
    emb = np.load(ed / "emb.npy", mmap_mode="r")
    idx_tr, idx_va, idx_te = (cache.indices_of(sf[k]) for k in ("train", "val", "test"))
    Xg, yg = GuardedArray(emb, forbidden=idx_te), GuardedArray(cache.labels, forbidden=idx_te)
    fit = s2_fit_fold(Xg, yg, idx_tr, idx_va, cfg["s2"], seed)         # fase fit: uji tidak terlihat
    assert not (Xg.accessed & set(map(int, idx_te))), "L2 dilanggar"
    thr = cfg["eval"]["threshold"]
    write_preds(d, "val", cache.meta, idx_va, s2_predict(fit, emb, idx_va), thr)
    write_preds(d, "test", cache.meta, idx_te, s2_predict(fit, emb, idx_te), thr)     # uji baru disentuh setelah model beku
    write_run_info(d, cfg, h, seed, {"rep": rep, "fold": fold, "cache_hash": cache.hash, "splits_hash": splits["meta"]["hash"],
                                     "device": "cpu", "C_terpilih": fit["C"], "val_loss_terpilih": fit["val_loss"],
                                     "n_train_images": fit["n_train"], "n_setelah_smote": fit["n_after_smote"],
                                     "sumber_pemilihan_C": "pasien validasi dalam lipatan latih"})
    mark_done(d, h)
    return str(d)


# ---------------------------------------------------------------- S3/S4: fitur PyRadiomics (radiomics_feats.py)
# Fungsi ekstraksi per-citra berada di modul 'larynx_rad' agar dapat dipakai proses anak (spawn, Windows).
RAD_SRC = r'''
import numpy as np
from scipy import ndimage as ndi

_EXT = None
_RAD = {}


def _extractor(rad):
    from radiomics import featureextractor

    settings = {"binWidth": rad["bin_width"], "force2D": True, "label": 1, "correctMask": True}
    ex = featureextractor.RadiomicsFeatureExtractor(**settings)
    ex.disableAllFeatures()
    for cls in ("firstorder", "shape2D", "glcm", "glrlm", "glszm", "gldm", "ngtdm"):
        ex.enableFeatureClassByName(cls)
    return ex


def _init(rad):
    global _EXT, _RAD
    import logging

    logging.getLogger("radiomics").setLevel(logging.ERROR)
    _RAD = rad
    _EXT = _extractor(rad)


def image_and_mask(img_rgb, roi, rad):
    gray = img_rgb[..., 1] if rad["channel"] == "green" else img_rgb.mean(axis=2)
    gray = gray.astype(np.float32)
    if rad["mask"] == "roi":
        mask = roi.astype(np.uint8)
    else:       # fov: piksel di luar area pandang bernilai 0 di cache
        mask = ndi.binary_fill_holes(img_rgb.max(axis=2) > 0).astype(np.uint8)
    return gray, mask


def features_of(args):
    img, roi = args
    import SimpleITK as sitk

    gray, mask = image_and_mask(img, roi, _RAD)
    if mask.sum() < 10:      # mask terlalu kecil -> fitur tak terdefinisi; NaN dicatat, tidak diisi diam-diam
        return {}
    r = _EXT.execute(sitk.GetImageFromArray(gray), sitk.GetImageFromArray(mask))
    return {k: float(v) for k, v in r.items() if k.startswith("original_")}
'''


def feature_dir(cfg: dict, cache) -> Path:
    h = config_hash({"rad": cfg["radiomics"], "cache": cache.hash})
    return P(cfg, "features", f"rad_{h[:12]}")


def extract_all(cfg: dict, cache_dir=None) -> Path:
    # Ekstraksi tidak memfit apa pun, jadi satu berkas fitur aman dipakai semua lipatan (L2). Jumlah fitur dicatat apa adanya.
    import multiprocessing as mp

    larynx_rad = install_module("larynx_rad", RAD_SRC)
    cache = Cache(cache_dir or find_cache(cfg))
    rad = cfg["radiomics"]
    h = config_hash({"rad": rad, "cache": cache.hash})
    d = feature_dir(cfg, cache)
    if is_done(d, h):
        print(f"[dilewati] fitur {d.name} sudah DONE")
        return d
    d.mkdir(parents=True, exist_ok=True)
    N = len(cache.meta)
    prog = d / "progress.json"
    done_n = json.loads(prog.read_text())["done"] if prog.exists() and (d / "feat.npy").exists() else 0
    names = json.loads((d / "names.json").read_text()) if (d / "names.json").exists() else None
    if done_n and names is not None and np.load(d / "feat.npy", mmap_mode="r").shape != (N, len(names)):
        print("[peringatan] feat.npy tidak cocok dengan progress.json/names.json: ekstraksi diulang dari awal")
        done_n = 0
    mm = None
    workers = rad["workers"] or max(1, (os.cpu_count() or 2) - 1)
    with mp.get_context("spawn").Pool(workers, initializer=larynx_rad._init, initargs=(rad,)) as pool:
        for s in range(done_n, N, rad["chunk"]):
            idx = range(s, min(s + rad["chunk"], N))
            rows = pool.map(larynx_rad.features_of, [(np.array(cache.images[i]), np.array(cache.roi[i])) for i in idx])
            if names is None:
                first = next((r for r in rows if r), None)
                if first is None:
                    raise RuntimeError("tidak ada fitur yang dapat diekstrak pada potongan pertama")
                names = sorted(first)
                atomic_write_json(d / "names.json", names)
            if mm is None:
                fresh = not (d / "feat.npy").exists() or done_n == 0
                mm = np.lib.format.open_memmap(d / "feat.npy", mode="w+" if fresh else "r+", dtype=np.float32, shape=(N, len(names)))
                if fresh:
                    mm[:] = np.nan       # baris belum terisi terlihat sebagai NaN, bukan nol
            for i, r in zip(idx, rows):
                mm[i] = [r.get(n, np.nan) for n in names]
            mm.flush()
            atomic_write_json(prog, {"done": idx[-1] + 1, "total": N})
            print(f"  radiomics {idx[-1] + 1}/{N}", flush=True)
    assert json.loads(prog.read_text())["done"] == N, "ekstraksi radiomik belum mencakup semua citra"
    n_nan = int(np.isnan(np.load(d / "feat.npy", mmap_mode="r")).any(axis=1).sum())
    atomic_write_json(d / "summary.json", {"n_features": len(names), "n_images": N, "n_images_with_nan": n_nan,
                                           "mask": rad["mask"], "channel": rad["channel"]})
    mark_done(d, h)
    return d


# ---------------------------------------------------------------- S3/S4: MinMax -> LASSO -> (CTGAN) -> SMOTE -> SVM/LDA (train_s3.py)
def _ctgan_augment(Xsel, y, c: dict, seed: int):
    from ctgan import CTGAN

    cols = [f"f{i}" for i in range(Xsel.shape[1])]
    df = pd.DataFrame(Xsel, columns=cols)
    df["y"] = y.astype(int)
    kw = {"epochs": c["epochs"], "batch_size": c["batch_size"], "verbose": False}
    if c["cuda"] != "auto":
        kw["cuda"] = bool(c["cuda"])
    gan = CTGAN(**kw)
    gan.fit(df, discrete_columns=["y"])
    syn = gan.sample(int(round(c["synth_fraction"] * len(df))))
    return syn[cols].to_numpy(), syn["y"].to_numpy().astype(int)


def s3_fit_fold(X, y, idx_train, idx_val, cfg: dict, seed: int) -> dict:
    s3 = cfg["s3"]
    Xtr, ytr = X[idx_train], y[idx_train]
    ok = ~np.isnan(Xtr).any(axis=1)         # citra tanpa fitur (mask kosong) tidak ikut melatih; jumlahnya dicatat
    Xtr, ytr = Xtr[ok], ytr[ok]
    scaler = MinMaxScaler().fit(Xtr)
    Xs = scaler.transform(Xtr)
    lasso = Lasso(alpha=s3["lasso_alpha"], max_iter=s3["lasso_max_iter"], random_state=seed).fit(Xs, ytr)
    sel = np.flatnonzero(np.abs(lasso.coef_) > 0)
    if sel.size == 0:
        raise ValueError(f"LASSO (alpha={s3['lasso_alpha']}) memilih 0 fitur; tidak diganti diam-diam. Periksa alpha/fitur.")
    Xsel = Xs[:, sel]
    n_ctgan = 0
    if s3["use_ctgan"]:
        Xg, yg = _ctgan_augment(Xsel, ytr, cfg["s4_ctgan"], seed)
        Xsel, ytr = np.vstack([Xsel, Xg]), np.concatenate([ytr, yg])
        n_ctgan = len(yg)
    k = max(1, min(s3["smote_k"], int(np.bincount(ytr).min()) - 1))
    Xres, yres = SMOTE(sampling_strategy="auto", k_neighbors=k, random_state=seed).fit_resample(Xsel, ytr)
    if s3["model"] == "svm":
        clf = SVC(kernel="linear", C=s3["svm_C"], probability=True, random_state=seed)
    elif s3["model"] == "lda":
        clf = LinearDiscriminantAnalysis()
    else:
        raise ValueError(f"s3.model tidak dikenal: {s3['model']}")
    clf.fit(Xres, yres)
    return {"scaler": scaler, "sel": sel, "clf": clf, "n_selected": int(sel.size), "n_train_dropped_nan": int((~ok).sum()),
            "n_ctgan": n_ctgan, "n_after_smote": int(len(yres))}


def s3_predict(fit: dict, X, idx) -> np.ndarray:
    Z = np.asarray(X[idx], dtype=np.float64)
    p = np.full(len(idx), 0.5)              # citra tanpa fitur: probabilitas netral 0,5, dihitung dan dicatat
    ok = ~np.isnan(Z).any(axis=1)
    if ok.any():
        p[ok] = fit["clf"].predict_proba(fit["scaler"].transform(Z[ok])[:, fit["sel"]])[:, 1]
    return p


def run_s3_job(cfg: dict, rep: int, fold: int, cache_dir=None) -> str:
    scenario = cfg["scenario"]
    splits = load_splits(cfg)
    check_l1(splits)
    cache = Cache(cache_dir or find_cache(cfg))
    fd = feature_dir(cfg, cache)
    d = job_dir(cfg, scenario, rep, fold)
    h = config_hash({"cfg": cfg, "rep": rep, "fold": fold, "cache": cache.hash, "splits": splits["meta"]["hash"]})
    if is_done(d, h):
        print(f"[dilewati] {scenario} rep{rep} fold{fold} sudah DONE")
        return str(d)
    if not (fd / "DONE").exists():
        raise FileNotFoundError("Fitur radiomik belum selesai (jalankan extract_all)")
    d.mkdir(parents=True, exist_ok=True)
    sf = get_fold(splits, rep, fold)
    atomic_write_json(d / "patients.json", {"train": sf["train"], "val": sf["val"], "test": sf["test"]})
    check_l3_job(json.loads((d / "patients.json").read_text(encoding="utf-8")), sf)
    seed = job_seed(cfg["seed"], scenario, rep, fold)
    set_seed(seed)
    feats = np.load(fd / "feat.npy", mmap_mode="r")
    idx_tr, idx_va, idx_te = (cache.indices_of(sf[k]) for k in ("train", "val", "test"))
    Xg, yg = GuardedArray(feats, forbidden=idx_te), GuardedArray(cache.labels, forbidden=idx_te)
    fit = s3_fit_fold(Xg, yg, idx_tr, idx_va, cfg, seed)
    assert not (Xg.accessed & set(map(int, idx_te))), "L2 dilanggar"
    thr = cfg["eval"]["threshold"]
    write_preds(d, "val", cache.meta, idx_va, s3_predict(fit, feats, idx_va), thr)
    write_preds(d, "test", cache.meta, idx_te, s3_predict(fit, feats, idx_te), thr)
    write_run_info(d, cfg, h, seed, {"rep": rep, "fold": fold, "cache_hash": cache.hash, "device": "cpu",
                                     "splits_hash": splits["meta"]["hash"], "lasso_alpha": cfg["s3"]["lasso_alpha"],
                                     "n_fitur_dipilih": fit["n_selected"], "n_latih_nan_dibuang": fit["n_train_dropped_nan"],
                                     "n_ctgan": fit["n_ctgan"], "n_setelah_smote": fit["n_after_smote"],
                                     "mask_radiomics": cfg["radiomics"]["mask"]})
    mark_done(d, h)
    return str(d)

In [ ]:
# ---------------------------------------------------------------- evaluasi tingkat pasien (dari evaluate.py Versi 1, FR-09, FR-10, FR-13)
# Probabilitas pasien = rata-rata citra; bootstrap persentil bertingkat 2.000 ulangan; selisih berpasangan; perbandingan UTAMA = S6 vs S1/S2.
from sklearn.metrics import roc_auc_score


METRICS = ["auroc", "f1", "recall", "precision", "accuracy"]
PRIMARY = {("S6", "S1"), ("S6", "S2")}


# ===================================================================== metrik dan bootstrap
def patient_metrics(y: np.ndarray, p: np.ndarray, thr: float) -> dict:
    pred = (p >= thr).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum())
    fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum())
    tn = int(((pred == 0) & (y == 0)).sum())
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    return {"auroc": float(roc_auc_score(y, p)) if len(np.unique(y)) == 2 else float("nan"),
            "f1": 2 * prec * rec / (prec + rec) if prec + rec else 0.0, "recall": rec, "precision": prec,
            "accuracy": (tp + tn) / len(y), "tp": tp, "fp": fp, "fn": fn, "tn": tn}


def _strata(y: np.ndarray) -> list[np.ndarray]:
    return [np.flatnonzero(y == c) for c in np.unique(y)]


def _boot_indices(strata: list[np.ndarray], rng: np.random.Generator) -> np.ndarray:
    return np.concatenate([rng.choice(s, size=len(s), replace=True) for s in strata])


def bootstrap_ci(y: np.ndarray, p: np.ndarray, thr: float, reps: int, level: float, seed: int) -> dict:
    est = patient_metrics(y, p, thr)
    rng = np.random.default_rng(seed)
    strata = _strata(y)
    draws = {m: [] for m in METRICS}
    for _ in range(reps):
        i = _boot_indices(strata, rng)
        r = patient_metrics(y[i], p[i], thr)
        for m in METRICS:
            draws[m].append(r[m])
    a = (1 - level) / 2 * 100
    out = {}
    for m in METRICS:
        lo, hi = np.nanpercentile(draws[m], [a, 100 - a])
        out[m] = {"estimate": est[m], "ci_low": float(lo), "ci_high": float(hi)}
    out["confusion"] = {k: est[k] for k in ("tp", "fp", "fn", "tn")}
    return out


def paired_bootstrap_diff(y, pa, pb, thr: float, reps: int, level: float, seed: int) -> dict:
    """Selisih A - B pada pasien yang sama. Bermakna bila interval tidak mencakup nol (PRD §3)."""
    ea, eb = patient_metrics(y, pa, thr), patient_metrics(y, pb, thr)
    rng = np.random.default_rng(seed)
    strata = _strata(y)
    d = {m: [] for m in METRICS}
    for _ in range(reps):
        i = _boot_indices(strata, rng)
        ra, rb = patient_metrics(y[i], pa[i], thr), patient_metrics(y[i], pb[i], thr)
        for m in METRICS:
            d[m].append(ra[m] - rb[m])
    a = (1 - level) / 2 * 100
    out = {}
    for m in METRICS:
        lo, hi = np.nanpercentile(d[m], [a, 100 - a])
        out[m] = {"diff": ea[m] - eb[m], "ci_low": float(lo), "ci_high": float(hi), "mencakup_nol": bool(lo <= 0 <= hi)}
    return out


def unpaired_bootstrap_diff(y1, p1, y2, p2, thr: float, reps: int, level: float, seed: int) -> dict:
    """Selisih kelompok 1 - kelompok 2 pada pasien BERBEDA (S7: dalam-distribusi vs pergeseran resolusi)."""
    e1, e2 = patient_metrics(y1, p1, thr), patient_metrics(y2, p2, thr)
    rng = np.random.default_rng(seed)
    s1, s2 = _strata(y1), _strata(y2)
    d = {m: [] for m in METRICS}
    for _ in range(reps):
        i, j = _boot_indices(s1, rng), _boot_indices(s2, rng)
        r1, r2 = patient_metrics(y1[i], p1[i], thr), patient_metrics(y2[j], p2[j], thr)
        for m in METRICS:
            d[m].append(r1[m] - r2[m])
    a = (1 - level) / 2 * 100
    out = {}
    for m in METRICS:
        lo, hi = np.nanpercentile(d[m], [a, 100 - a])
        out[m] = {"diff": e1[m] - e2[m], "ci_low": float(lo), "ci_high": float(hi), "mencakup_nol": bool(lo <= 0 <= hi)}
    return out


# ===================================================================== agregasi job
def collect_predictions(cfg: dict, scenario: str, jobs_root=None, allow_partial: bool = False) -> tuple[pd.DataFrame, dict]:
    """Baca preds_test_patients.csv dari job berstatus DONE. Tanpa allow_partial, semua job wajib selesai."""
    splits = load_splits(cfg)
    rows, expected, found = [], 0, 0
    for r in splits["repeats"]:
        for f in r["folds"]:
            expected += 1
            d = job_dir(cfg, scenario, r["rep"], f["fold"]) if jobs_root is None else Path(jobs_root) / scenario / f"rep{r['rep']}_fold{f['fold']}"
            if (d / "DONE").exists() and (d / "preds_test_patients.csv").exists():
                df = pd.read_csv(d / "preds_test_patients.csv")
                df["rep"], df["fold"] = r["rep"], f["fold"]
                rows.append(df)
                found += 1
    if found < expected and not allow_partial:
        raise RuntimeError(f"{scenario}: hanya {found}/{expected} job DONE; agregasi hanya membaca job selesai (PRD §8)")
    if not rows:
        raise RuntimeError(f"{scenario}: tidak ada job DONE")
    return pd.concat(rows, ignore_index=True), {"jobs_found": found, "jobs_expected": expected}


def aggregate_patients(preds: pd.DataFrame, average_repeats: bool = True) -> pd.DataFrame:
    # average_repeats=True (D4): probabilitas pasien dirata-ratakan atas semua ulangan.
    # False: hanya ulangan pertama yang dipakai (satu prediksi per pasien); n_ulangan tetap dilaporkan = 1.
    if not average_repeats:
        preds = preds[preds["rep"] == preds["rep"].min()]
    g = preds.groupby("patient_uid")
    return g.agg(label=("label", "first"), prob=("prob_malignant", "mean"), n_ulangan=("rep", "nunique")).reset_index()


def fold_table(preds: pd.DataFrame, thr: float) -> pd.DataFrame:
    rows = []
    for (rep, fold), g in preds.groupby(["rep", "fold"]):
        m = patient_metrics(g.label.to_numpy(), g.prob_malignant.to_numpy(), thr)
        rows.append({"rep": rep, "fold": fold, "n_pasien": len(g), "n_ganas": int(g.label.sum()), **m})
    return pd.DataFrame(rows)


def evaluate_preds(cfg: dict, scenario: str, preds: pd.DataFrame, cov: dict, extra_note: str = "") -> dict:
    ev = cfg["eval"]
    pat = aggregate_patients(preds, ev["average_repeats"])
    y, p = pat.label.to_numpy(), pat.prob.to_numpy()
    res = bootstrap_ci(y, p, ev["threshold"], ev["bootstrap_reps"], ev["ci_level"], ev["bootstrap_seed"])
    note = "PARSIAL: tidak semua job selesai" if cov["jobs_found"] < cov["jobs_expected"] else \
        "lengkap; interval = bootstrap persentil bertingkat atas pasien"
    rows = [{"scenario": scenario, "metric": m, "estimate": res[m]["estimate"], "ci_low": res[m]["ci_low"],
             "ci_high": res[m]["ci_high"], "n_pasien": len(pat), "n_ganas": int(y.sum()),
             "job_selesai": f"{cov['jobs_found']}/{cov['jobs_expected']}", "catatan": (note + " " + extra_note).strip()}
            for m in METRICS]
    out_dir = P(cfg, "results")
    atomic_write_bytes(out_dir / f"results_{scenario}.csv", pd.DataFrame(rows).to_csv(index=False).encode("utf-8"))
    atomic_write_bytes(out_dir / f"fold_metrics_{scenario}.csv", fold_table(preds, ev["threshold"]).to_csv(index=False).encode("utf-8"))
    atomic_write_bytes(out_dir / f"patient_probs_{scenario}.csv", pat.to_csv(index=False).encode("utf-8"))
    return {"scenario": scenario, "coverage": cov, "table": pd.DataFrame(rows), "confusion": res["confusion"], "patients": pat}


def evaluate_scenario(cfg: dict, scenario: str, allow_partial: bool = False) -> dict:
    preds, cov = collect_predictions(cfg, scenario, allow_partial=allow_partial)
    return evaluate_preds(cfg, scenario, preds, cov)


def load_result(cfg: dict, scenario: str) -> dict:
    return {"scenario": scenario, "patients": pd.read_csv(P(cfg, "results", f"patient_probs_{scenario}.csv"))}


def compare_scenarios(cfg: dict, a: dict, b: dict, label: str | None = None) -> pd.DataFrame:
    """Selisih berpasangan A - B. Label otomatis: 'utama' hanya untuk S6 vs S1/S2; selain itu 'eksploratif'."""
    ev = cfg["eval"]
    label = label or ("utama" if (a["scenario"], b["scenario"]) in PRIMARY else "eksploratif")
    m = a["patients"].merge(b["patients"], on="patient_uid", suffixes=("_a", "_b"))
    assert (m.label_a == m.label_b).all(), "label pasien tidak konsisten antar skenario"
    d = paired_bootstrap_diff(m.label_a.to_numpy(), m.prob_a.to_numpy(), m.prob_b.to_numpy(), ev["threshold"],
                              ev["bootstrap_reps"], ev["ci_level"], ev["bootstrap_seed"])
    rows = [{"A": a["scenario"], "B": b["scenario"], "metric": k, "selisih_A_minus_B": v["diff"], "ci_low": v["ci_low"],
             "ci_high": v["ci_high"], "mencakup_nol": v["mencakup_nol"], "n_pasien": len(m), "status": label,
             "utama_untuk_metrik": k in ("auroc", "f1") if label == "utama" else False} for k, v in d.items()]
    df = pd.DataFrame(rows)
    atomic_write_bytes(P(cfg, "results", f"compare_{a['scenario']}_vs_{b['scenario']}.csv"), df.to_csv(index=False).encode("utf-8"))
    return df


# ===================================================================== rasio sintetis (G2, L4)
def _min_val_loss(cfg: dict, scenario: str, rep: int, fold: int) -> float | None:
    h = job_dir(cfg, scenario, rep, fold) / "history.csv"
    if not (job_dir(cfg, scenario, rep, fold) / "DONE").exists() or not h.exists():
        return None
    return float(pd.read_csv(h).val_loss_balanced.min())


def assemble_selected(cfg: dict, base: str, ratios: list[float], allow_partial: bool = False) -> tuple[pd.DataFrame, dict]:
    """Untuk tiap (rep, lipatan) pilih rasio dengan val_loss_balanced terendah (pasien VALIDASI), ambil prediksi UJI-nya."""
    splits = load_splits(cfg)
    rows, sel, expected, found = [], [], 0, 0
    for r in splits["repeats"]:
        for f in r["folds"]:
            expected += 1
            vals = {ra: _min_val_loss(cfg, f"{base}_r{int(round(ra * 100))}", r["rep"], f["fold"]) for ra in ratios}
            ok = {k: v for k, v in vals.items() if v is not None}
            if len(ok) < len(ratios) and not allow_partial:
                raise RuntimeError(f"{base} rep{r['rep']} fold{f['fold']}: hanya {len(ok)}/{len(ratios)} rasio selesai")
            if not ok:
                continue
            best = min(ok, key=lambda k: (ok[k], k))
            d = job_dir(cfg, f"{base}_r{int(round(best * 100))}", r["rep"], f["fold"])
            df = pd.read_csv(d / "preds_test_patients.csv")
            df["rep"], df["fold"], df["ratio"] = r["rep"], f["fold"], best
            rows.append(df)
            sel.append({"rep": r["rep"], "fold": f["fold"], "ratio_terpilih": best,
                        **{f"val_loss_r{int(round(k * 100))}": v for k, v in ok.items()},
                        "sumber_pemilihan": "val_loss_balanced pasien validasi dalam lipatan latih"})
            found += 1
    if not rows:
        raise RuntimeError(f"{base}: tidak ada job rasio yang DONE")
    atomic_write_bytes(P(cfg, "results", f"ratio_selection_{base}.csv"), pd.DataFrame(sel).to_csv(index=False).encode("utf-8"))
    return pd.concat(rows, ignore_index=True), {"jobs_found": found, "jobs_expected": expected}


def evaluate_selected(cfg: dict, base: str, allow_partial: bool = False) -> dict:
    ratios = cfg["gan"]["ratios"]
    preds, cov = assemble_selected(cfg, base, ratios, allow_partial)
    return evaluate_preds(cfg, base, preds, cov, extra_note="rasio dipilih per lipatan dari validasi (G2, L4)")


def evaluate_all_ratios(cfg: dict, base: str, allow_partial: bool = False) -> pd.DataFrame:
    """Semua rasio dilaporkan (G2: r=25%, 50%, 100% semuanya)."""
    out = []
    for ra in cfg["gan"]["ratios"]:
        r = evaluate_scenario(cfg, f"{base}_r{int(round(ra * 100))}", allow_partial)
        out.append(r["table"])
    return pd.concat(out, ignore_index=True)


def select_best_config(cfg: dict, bases: tuple[str, ...] = ("S5", "S6")) -> dict:
    """INFORMASI untuk CV utama (rasio dengan rata-rata val_loss_balanced terendah). S7 TIDAK memakainya: validasi CV utama
    melibatkan pasien resolusi minor yang menjadi pasien uji S7 (L5). Lihat select_s7_ratio."""
    splits = load_splits(cfg, "main")
    out = {}
    for base in bases:
        means = {}
        for ra in cfg["gan"]["ratios"]:
            v = [_min_val_loss(cfg, f"{base}_r{int(round(ra * 100))}", r["rep"], f["fold"]) for r in splits["repeats"] for f in r["folds"]]
            if any(x is None for x in v):
                raise RuntimeError(f"{base}_r{int(round(ra * 100))}: ada job belum DONE; konfigurasi terbaik butuh semua lipatan")
            means[ra] = float(np.mean(v))
        best = min(means, key=lambda k: (means[k], k))
        out[base] = {"ratio": best, "mean_val_loss_balanced": {str(k): v for k, v in means.items()},
                     "source": "val_loss_balanced pasien validasi dalam lipatan latih (L4); uji tidak dipakai"}
    sel = P(cfg, "cp4", "lambda_selection.json")
    if sel.exists() and "S6" in out:
        out["S6"]["lambda_roi"] = json.loads(sel.read_text(encoding="utf-8"))["chosen"]
    atomic_write_json(P(cfg, "results", "best_config.json"), out)
    return out


# ===================================================================== S7
def select_s7_ratio(cfg: dict, base: str, allow_partial: bool = False) -> dict:
    """L4/L5: rasio S7 dipilih HANYA dari val_loss_balanced pasien validasi di dalam split S7 (resolusi mayoritas),
    dirata-ratakan atas ulangan S7. Satu rasio untuk semua ulangan. Pasien uji S7 (resolusi lain) tidak ikut."""
    splits = load_splits(cfg, "s7")
    rows, means = [], {}
    for ra in cfg["gan"]["ratios"]:
        v = [(r["rep"], _min_val_loss(cfg, f"{base}_r{int(round(ra * 100))}", r["rep"], f["fold"])) for r in splits["repeats"] for f in r["folds"]]
        ok = [x for _, x in v if x is not None]
        if len(ok) < len(v) and not allow_partial:
            raise RuntimeError(f"{base}_r{int(round(ra * 100))}: hanya {len(ok)}/{len(v)} job DONE; pemilihan rasio S7 butuh semuanya")
        if ok:
            means[ra] = float(np.mean(ok))
        rows += [{"ratio": ra, "rep": rp, "val_loss_balanced": x} for rp, x in v]
    if not means:
        raise RuntimeError(f"{base}: tidak ada job rasio S7 yang DONE")
    best = min(means, key=lambda k: (means[k], k))
    out = {"base": base, "ratio": best, "mean_val_loss_balanced": {str(k): v for k, v in means.items()},
           "source": "val_loss_balanced pasien validasi DI DALAM split S7 (resolusi mayoritas); pasien uji S7 dan CV utama tidak dipakai (L5)"}
    atomic_write_bytes(P(cfg, "results", f"ratio_selection_{base}.csv"), pd.DataFrame(rows).to_csv(index=False).encode("utf-8"))
    atomic_write_json(P(cfg, "results", f"best_ratio_{base}.json"), out)
    return out


def evaluate_s7_selected(cfg: dict, base: str, allow_partial: bool = False) -> dict:
    sel = select_s7_ratio(cfg, base, allow_partial)
    ra = sel["ratio"]
    preds, cov = collect_predictions(cfg, f"{base}_r{int(round(ra * 100))}", allow_partial=allow_partial)
    return evaluate_preds(cfg, base, preds, cov, extra_note=f"rasio {ra} dipilih dari validasi dalam split S7 (L4, L5)")


def s7_gap(cfg: dict, main_scenario: str, s7_scenario: str) -> pd.DataFrame:
    """Selisih kinerja dalam-distribusi (pasien resolusi mayoritas pada CV utama) vs pergeseran (uji gabungan S7).
    Pasien berbeda dan kondisi pelatihan tidak persis sama (CV utama juga melatih pada resolusi lain) -> EKSPLORATIF."""
    ev = cfg["eval"]
    audit = pd.read_csv(P(cfg, "cp1", "audit.csv"))
    major = load_splits(cfg, "s7")["major_resolution"]
    a = pd.read_csv(P(cfg, "results", f"patient_probs_{main_scenario}.csv")).merge(audit[["patient_uid", "resolution"]], on="patient_uid")
    ind = a[a.resolution == major]
    shifted = pd.read_csv(P(cfg, "results", f"patient_probs_{s7_scenario}.csv"))
    d = unpaired_bootstrap_diff(ind.label.to_numpy(), ind.prob.to_numpy(), shifted.label.to_numpy(), shifted.prob.to_numpy(),
                                ev["threshold"], ev["bootstrap_reps"], ev["ci_level"], ev["bootstrap_seed"])
    e_in = patient_metrics(ind.label.to_numpy(), ind.prob.to_numpy(), ev["threshold"])
    e_sh = patient_metrics(shifted.label.to_numpy(), shifted.prob.to_numpy(), ev["threshold"])
    rows = [{"metode": main_scenario, "metric": m, "dalam_distribusi": e_in[m], "pergeseran_S7": e_sh[m],
             "selisih": v["diff"], "ci_low": v["ci_low"], "ci_high": v["ci_high"], "mencakup_nol": v["mencakup_nol"],
             "n_dalam": len(ind), "n_pergeseran": len(shifted), "status": "eksploratif"} for m, v in d.items()]
    df = pd.DataFrame(rows)
    atomic_write_bytes(P(cfg, "results", f"s7_gap_{main_scenario}.csv"), df.to_csv(index=False).encode("utf-8"))
    return df


# ===================================================================== tabel (FR-13)
ORDER = ["S1", "S2", "S3_svm", "S3_lda", "S4_svm", "S4_lda", "S5", "S6", "S7_S1", "S7_S5", "S7_S6",
         "S3_svm_a01", "S3_lda_a01", "S4_svm_a01", "S4_lda_a01"]


def make_tables(cfg: dict) -> dict:
    res = []
    for sc in ORDER + [f"{b}_r{int(round(r * 100))}" for b in ("S5", "S6") for r in cfg["gan"]["ratios"]]:
        p = P(cfg, "results", f"results_{sc}.csv")
        if p.exists():
            res.append(pd.read_csv(p))
    if not res:
        raise RuntimeError("belum ada results_*.csv")
    allr = pd.concat(res, ignore_index=True)
    allr["sel"] = allr.apply(lambda r: f"{r.estimate:.3f} [{r.ci_low:.3f}, {r.ci_high:.3f}]", axis=1)
    wide = allr.pivot(index="scenario", columns="metric", values="sel")[METRICS]
    meta = allr.groupby("scenario").agg(n_pasien=("n_pasien", "first"), job_selesai=("job_selesai", "first"),
                                        catatan=("catatan", "first"))
    t33 = wide.join(meta)
    t33["status"] = ["eksploratif (sensitivitas LASSO)" if s.endswith("_a01") else
                     ("rincian rasio" if "_r" in s else "") for s in t33.index]
    t33 = t33.reindex([s for s in ORDER if s in t33.index] + [s for s in t33.index if s not in ORDER])
    atomic_write_bytes(P(cfg, "results", "tabel_3_3.csv"), t33.reset_index().to_csv(index=False).encode("utf-8"))
    atomic_write_bytes(P(cfg, "results", "tabel_3_3.md"), _df_to_md(t33.reset_index()).encode("utf-8"))

    lit = yaml.safe_load(open(ROOT / "configs" / "literature.yaml", encoding="utf-8"))["rows"]
    t34 = pd.DataFrame(lit)
    ours = []
    for sc in ("S1", "S2", "S6"):
        r = allr[allr.scenario == sc]
        if not r.empty:
            f1 = r[r.metric == "f1"].iloc[0]
            ours.append({"sumber": f"Penelitian ini ({sc})", "data": "CE-NBI 210 pasien, CV 5x2 per pasien",
                         "metrik": "F1 pasien [CI 95%]", "nilai_validasi_silang": f"{f1.estimate:.3f} [{f1.ci_low:.3f}, {f1.ci_high:.3f}]",
                         "nilai_uji": None, "catatan": "interval = bootstrap pasien; bukan median fold seperti Wang dkk."})
    t34 = pd.concat([t34, pd.DataFrame(ours)], ignore_index=True)
    atomic_write_bytes(P(cfg, "results", "tabel_3_4.csv"), t34.to_csv(index=False).encode("utf-8"))
    return {"tabel_3_3": t33, "tabel_3_4": t34}


def _df_to_md(df: pd.DataFrame) -> str:
    # Tabel Markdown sungguhan, dengan atau tanpa tabulate (versi lama menulis CSV ke berkas .md bila tabulate tidak ada).
    cells = [[str(c) for c in df.columns]] + [[("" if pd.isna(v) else str(v)) for v in row] for row in df.itertuples(index=False)]
    head, rows = cells[0], cells[1:]
    return "\n".join(["| " + " | ".join(head) + " |", "| " + " | ".join("---" for _ in head) + " |"] + ["| " + " | ".join(r) + " |" for r in rows]) + "\n"


def _has_tabulate() -> bool:
    try:
        import tabulate  # noqa: F401
        return True
    except ImportError:
        return False


def run_comparisons(cfg: dict) -> pd.DataFrame:
    """Perbandingan utama (S6 vs S1, S6 vs S2) + eksploratif lain yang hasilnya tersedia."""
    pairs = [("S6", "S1"), ("S6", "S2"), ("S6", "S5"), ("S5", "S1"), ("S1", "S2"), ("S6", "S3_svm"), ("S6", "S3_lda"),
             ("S6", "S4_svm"), ("S6", "S4_lda")]
    out = []
    for a, b in pairs:
        if (P(cfg, "results", f"patient_probs_{a}.csv").exists() and P(cfg, "results", f"patient_probs_{b}.csv").exists()):
            out.append(compare_scenarios(cfg, load_result(cfg, a), load_result(cfg, b)))
    if not out:
        raise RuntimeError("belum ada pasangan skenario yang hasilnya lengkap")
    df = pd.concat(out, ignore_index=True)
    atomic_write_bytes(P(cfg, "results", "comparisons_all.csv"), df.to_csv(index=False).encode("utf-8"))
    return df

In [ ]:
# ---------------------------------------------------------------- pemeriksaan checkpoint (dari checkpoint.py Versi 1)
# Prinsip: item yang tidak bisa dijalankan dilaporkan apa adanya (GAGAL / TIDAK_TERSEDIA / MENUNGGU), tidak pernah diubah menjadi LULUS.
# `lulus` benar hanya bila tidak ada GAGAL atau MENUNGGU. manifest.json ditulis hanya bila lulus. Tag git TIDAK dibuat di sini.
def _scenario_items(cfg: dict, sc: str, label=None):
    # Item standar satu skenario: semua job DONE, L3 lolos, tabel hasil lengkap dengan interval.
    label = label or sc
    items, files = [], []
    st = job_status(cfg, sc)
    n_done = sum(r["DONE"] for r in st)
    items.append(_item(f"{label}: seluruh lipatan berstatus DONE", LULUS if n_done == len(st) else GAGAL, f"{n_done}/{len(st)}"))
    try:
        n = check_l3_dir(P(cfg, "jobs"), sc, load_splits(cfg))
        items.append(_item(f"{label}: L3 daftar pasien tiap job", LULUS, f"{n} job"))
    except LeakageError as e:
        items.append(_item(f"{label}: L3", GAGAL, str(e)))
    rp = P(cfg, "results", f"results_{sc}.csv")
    if rp.exists():
        r = pd.read_csv(rp)
        full = r.job_selesai.str.split("/").map(lambda a: a[0] == a[1]).all()
        items.append(_item(f"{label}: tabel hasil dengan interval kepercayaan", LULUS if full else GAGAL,
                           "; ".join(f"{x.metric}={x.estimate:.3f} [{x.ci_low:.3f}, {x.ci_high:.3f}]" for x in r.itertuples())
                           + f" | job {r.job_selesai.iloc[0]}"))
        files += [rp, P(cfg, "results", f"fold_metrics_{sc}.csv"), P(cfg, "results", f"patient_probs_{sc}.csv")]
    else:
        items.append(_item(f"{label}: tabel hasil", GAGAL, f"{rp.name} tidak ada"))
    return items, files


CP3_SCENARIOS = {"S1": "configs/s1.yaml", "S2": "configs/s2.yaml", "S3_svm": "configs/s3_svm.yaml", "S3_lda": "configs/s3_lda.yaml",
                 "S4_svm": "configs/s4_svm.yaml", "S4_lda": "configs/s4_lda.yaml"}
CP3_SENSITIVITY = {"S3_svm_a01": "configs/s3_svm_a01.yaml", "S3_lda_a01": "configs/s3_lda_a01.yaml",
                   "S4_svm_a01": "configs/s4_svm_a01.yaml", "S4_lda_a01": "configs/s4_lda_a01.yaml"}


def finalize_cp3(cfg: dict) -> dict:
    out = P(cfg, "cp3")
    items, files = [], []
    for sc, path in CP3_SCENARIOS.items():
        it, fl = _scenario_items(load_merged(path), sc)
        items += it
        files += fl
    cmp_ = P(cfg, "results", "compare_S1_vs_S2.csv")
    items.append(_item("Perbandingan S1 vs S2 (berlabel eksploratif)", LULUS if cmp_.exists() else GAGAL, "ada" if cmp_.exists() else "belum dibuat"))
    files.append(cmp_)
    for sc, path in CP3_SENSITIVITY.items():        # informasi: analisis sensitivitas D3 (eksploratif), tidak menjadi gerbang
        c = load_merged(path)
        n = sum(r["DONE"] for r in job_status(c, sc))
        items.append(_item(f"{sc} (sensitivitas LASSO 0,01, eksploratif; informasi)", LULUS, f"{n}/{len(job_status(c, sc))} job DONE"))
    fd = sorted(P(cfg, "features").glob("rad_*/summary.json")) if P(cfg, "features").exists() else []
    if fd:
        sm = json.loads(fd[-1].read_text(encoding="utf-8"))
        items.append(_item("Fitur radiomik diekstrak (jumlah dilaporkan apa adanya)", LULUS,
                           f"{sm['n_features']} fitur (Wang dkk.: 104), mask={sm['mask']}, citra dengan NaN={sm['n_images_with_nan']}"))
    else:
        items.append(_item("Fitur radiomik", GAGAL, "summary.json tidak ada"))
    for d in ("D3", "D4", "D7"):
        items.append(_decision_item(d))
    lg = P(cfg, "jobs", "launch.log")
    items.append(_item("Penjalan job teruji pada beban nyata", LULUS if lg.exists() else GAGAL, "launch.log ada" if lg.exists() else "belum ada job yang dijalankan"))
    items.append(_review_item(out))
    return _finish("CP3", out, items, files)

## 5. Daftar skenario

In [ ]:
REPS, FOLDS = None, None      # None = semua. Membatasi JOB YANG DILUNCURKAN saja, bukan memilih hasil; gerbang butuh semuanya.
ALLOW_PARTIAL = REPS is not None or FOLDS is not None      # tabel parsial otomatis ditandai PARSIAL dan tidak memenuhi gerbang

MAIN = [('S2', 'configs/s2.yaml', run_s2_job), ('S3_svm', 'configs/s3_svm.yaml', run_s3_job),
        ('S3_lda', 'configs/s3_lda.yaml', run_s3_job), ('S4_svm', 'configs/s4_svm.yaml', run_s3_job),
        ('S4_lda', 'configs/s4_lda.yaml', run_s3_job), ('S1', 'configs/s1.yaml', run_swin_job)]      # S1 terakhir (paling lama)
SENS = [('S3_svm_a01', 'configs/s3_svm_a01.yaml', run_s3_job), ('S3_lda_a01', 'configs/s3_lda_a01.yaml', run_s3_job),
        ('S4_svm_a01', 'configs/s4_svm_a01.yaml', run_s3_job), ('S4_lda_a01', 'configs/s4_lda_a01.yaml', run_s3_job)]
RUN_SENSITIVITY = True      # analisis sensitivitas D3 (eksploratif)

bm = json.loads(P(cfg, 'cp0', 'benchmark.json').read_text(encoding='utf-8'))
print('Benchmark CP0:', round(bm['estimated_epoch_minutes'], 1), 'menit/epoch ->', round(bm['estimated_30_epoch_hours_per_job'], 1), 'jam per job S1 (batas atas)')

## 6. Fitur untuk S2 dan S3/S4 (dihitung sekali; dapat dilanjutkan)

In [ ]:
run_embed(load_merged('configs/s2.yaml'))
extract_all(load_merged('configs/s3_svm.yaml'))

## 7. Jalankan job (`n_splits × n_repeats` per skenario)

In [ ]:
for sc, path, fn in MAIN:
    print('=== ', sc)
    launch_scenario(path, fn, REPS, FOLDS)
    display(pd.DataFrame(job_status(load_merged(path), sc)).T)

## 8. Analisis sensitivitas LASSO 0,01 (eksploratif, D3)

In [ ]:
if RUN_SENSITIVITY:
    for sc, path, fn in SENS:
        print('=== ', sc)
        launch_scenario(path, fn, REPS, FOLDS)

## 9. Evaluasi tingkat pasien (AUROC, F1, recall, presisi, akurasi) dengan bootstrap persentil bertingkat 2.000 ulangan

In [ ]:
R = {}
for sc, path, _ in MAIN + (SENS if RUN_SENSITIVITY else []):
    R[sc] = evaluate_scenario(load_merged(path), sc, allow_partial=ALLOW_PARTIAL)
display(pd.concat([r['table'].assign(skenario=sc)[['skenario', 'metric', 'estimate', 'ci_low', 'ci_high', 'job_selesai']] for sc, r in R.items()]))

## 10. Selisih berpasangan S1 − S2 (**eksploratif**; perbandingan utama PRD adalah S6 lawan S1 dan S2)

In [ ]:
display(compare_scenarios(cfg, R['S1'], R['S2'], label='eksploratif'))

## 11. Pemeriksaan checkpoint CP3 dan `manifest.json` (sel terakhir)

In [ ]:
# Isi nama peninjau KEDUA setelah benar-benar meninjau hasil di atas (RULES §3), lalu jalankan sel ini.
REVIEWER = None      # contoh: 'Ahmad'
if REVIEWER:
    approve_review('CP3', REVIEWER)

In [ ]:
report = finalize_cp3(cfg)
assert report['checkpoint'] == 'CP3'